# 2. ETRI 위키백과 QA API 실험 - 외부 데이터셋 활용

본 노트북은 **ETRI 위키백과 QA API**를 활용하여 외부 데이터셋을 수집하고 MRC 모델을 학습/평가합니다.

**ETRI 위키백과 QA API 정보:**
- API URL: `http://epretx.etri.re.kr:8000/api/WikiQA/`
- 참고 문서: https://epretx.etri.re.kr/apiDetail?id=62
- 일일 호출 제한: 5,000건/일

**실험 목표:**
- ETRI 외부 데이터셋의 품질 확인
- 외부 데이터만으로 학습한 모델의 성능 평가
- 기존 데이터셋과의 성능 비교 기반 마련


## 2.1. 환경 설정 및 라이브러리 Import


In [43]:
# 필요한 패키지 설치
%pip install --upgrade accelerate urllib3


Note: you may need to restart the kernel to use updated packages.


In [44]:
# OpenMP 충돌 방지 설정 (Windows 환경)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

# 필요한 라이브러리 import
import sys
import json
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datasets import Dataset, DatasetDict, load_from_disk
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import re
import math
import evaluate
from dataclasses import dataclass, field
import urllib3
import time

# Transformers
from transformers import (
    AutoConfig,
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    TrainingArguments,
    Trainer,
    set_seed,
)

# 프로젝트 루트 경로 설정
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

# src 모듈 import
from src.config import DataTrainingArguments, ModelArguments
from src.training.trainer_qa import QuestionAnsweringTrainer
from src.utils import postprocess_qa_predictions

# 시각화 설정
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style("whitegrid")

# 재현성을 위한 시드 설정
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"프로젝트 루트: {project_root}")
print(f"사용 디바이스: {device}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


프로젝트 루트: D:\Repos\pro-nlp-mrc-nlp-01
사용 디바이스: cuda
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 2070


In [45]:
# 한글 폰트 설정
import matplotlib
import platform

def set_korean_font():
    system = platform.system()
    if system == 'Windows':
        font_name = 'Malgun Gothic'
    elif system == 'Darwin':  # Mac
        font_name = 'AppleGothic'
    else:  # Linux
        font_name = 'NanumGothic'
    matplotlib.rc('font', family=font_name)
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()


## 2.2. ETRI 위키백과 QA API 설정


In [46]:
# ETRI API 설정
# ⚠️ 중요: 아래 YOUR_ACCESS_KEY를 발급받은 API 키로 교체하세요!
# API 키 발급: https://epretx.etri.re.kr 에서 회원가입 후 발급

ETRI_API_URL = "http://epretx.etri.re.kr:8000/api/WikiQA/"
ETRI_ACCESS_KEY = "d4a79a0b-64a4-432a-8a9a-24fd5bf433e4"

# 데이터 저장 경로
data_dir = Path().resolve() / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 실험 결과 저장 경로
experiment_dir = Path().resolve() / "experiments" / "etri_only"
experiment_dir.mkdir(parents=True, exist_ok=True)

print(f"데이터 저장 경로: {data_dir}")
print(f"실험 결과 저장 경로: {experiment_dir}")


데이터 저장 경로: D:\Repos\pro-nlp-mrc-nlp-01\notebooks\external_data_set\data
실험 결과 저장 경로: D:\Repos\pro-nlp-mrc-nlp-01\notebooks\external_data_set\experiments\etri_only


In [47]:
class ETRIWikiQA:
    """ETRI 위키백과 QA API 클라이언트"""
    
    def __init__(self, access_key: str):
        """
        Args:
            access_key: ETRI API 접근 키
        """
        self.api_url = ETRI_API_URL
        self.access_key = access_key
        # SSL 인증서 검증 비활성화 (Windows 환경 호환성)
        self.http = urllib3.PoolManager(cert_reqs='CERT_NONE')
        # 경고 비활성화
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    def query(self, question: str, engine_type: str = "hybridqa") -> Dict:
        """
        위키백과 QA API 호출
        
        Args:
            question: 질문 텍스트
            engine_type: 엔진 타입 (irqa, kbqa, hybridqa)
                - irqa: 언어분석 + 기계독해 기반
                - kbqa: 지식베이스 기반
                - hybridqa: irqa와 kbqa 통합
        
        Returns:
            API 응답 결과
        """
        request_json = {
            "argument": {
                "question": question,
                "type": engine_type
            }
        }
        
        try:
            response = self.http.request(
                "POST",
                self.api_url,
                headers={
                    "Content-Type": "application/json; charset=UTF-8",
                    "Authorization": self.access_key
                },
                body=json.dumps(request_json)
            )
            
            if response.status == 200:
                return json.loads(response.data.decode('utf-8'))
            else:
                # 403 에러 시 상세 원인 출력
                error_body = response.data.decode('utf-8')
                print(f"API 오류 [{response.status}]: {error_body}")
                
                # 403 에러 원인 안내
                if response.status == 403:
                    print("\n⚠️ 403 오류 원인 확인:")
                    print("  - Empty Auth Header: API 키가 헤더에 없음")
                    print("  - Invalid Key: API 키가 유효하지 않음")
                    print("  - Daily Limit Exceeded: 일일 호출 제한(5,000건) 초과")
                    print("  - Not Allowed IP: 허용되지 않은 IP")
                    print("\n해결 방법:")
                    print("  1. https://epretx.etri.re.kr 에서 API 키 재발급")
                    print("  2. ETRI_ACCESS_KEY 변수에 올바른 키 입력")
                    print("  3. API 설정에서 IP 허용 확인")
                return None
                
        except Exception as e:
            print(f"요청 실패: {e}")
            return None
    
    def extract_qa_data(self, response: Dict) -> Optional[Dict]:
        """
        API 응답에서 QA 데이터 추출
        
        Args:
            response: API 응답
            
        Returns:
            추출된 QA 데이터 (question, answer, context, confidence)
        """
        if not response or response.get('result') != 0:
            return None
        
        try:
            return_object = response.get('return_object', {})
            
            # WiKiInfo 안에 IRInfo와 AnswerInfo가 있음
            wiki_info = return_object.get('WiKiInfo', {})
            
            # 정답 정보 추출
            answer_info = wiki_info.get('AnswerInfo', [])
            if not answer_info:
                return None
            
            best_answer = answer_info[0]
            answer = best_answer.get('answer', '')
            confidence = best_answer.get('confidence', 0)
            
            # 검색 정보 추출 (context)
            ir_info = wiki_info.get('IRInfo', [])
            context = ""
            wiki_title = ""
            if ir_info:
                context = ir_info[0].get('sent', '')
                wiki_title = ir_info[0].get('wiki_title', '')
            
            return {
                'answer': answer,
                'confidence': confidence,
                'context': context,
                'wiki_title': wiki_title
            }
            
        except Exception as e:
            print(f"데이터 추출 실패: {e}")
            return None


# API 클라이언트 초기화
etri_qa = ETRIWikiQA(ETRI_ACCESS_KEY)
print("ETRI 위키백과 QA API 클라이언트 초기화 완료")


ETRI 위키백과 QA API 클라이언트 초기화 완료


## 2.3. ETRI API 테스트


In [48]:
# API 테스트 (실제 API 키가 있을 때만 실행)
if ETRI_ACCESS_KEY != "YOUR_ACCESS_KEY":
    print(f"✓ API 키 설정됨: {ETRI_ACCESS_KEY[:10]}..." if len(ETRI_ACCESS_KEY) > 10 else f"✓ API 키 설정됨: {ETRI_ACCESS_KEY}")
    print(f"✓ API URL: {ETRI_API_URL}")
    print("-" * 50)
    
    test_question = "대한민국의 수도는 어디인가요?"
    print(f"테스트 질문: {test_question}")
    print("-" * 50)
    
    response = etri_qa.query(test_question, engine_type="hybridqa")
    
    if response:
        print("✅ API 호출 성공!")
        qa_data = etri_qa.extract_qa_data(response)
        if qa_data:
            print(f"정답: {qa_data['answer']}")
            print(f"신뢰도: {qa_data['confidence']:.4f}")
            print(f"위키 제목: {qa_data['wiki_title']}")
            print(f"문맥: {qa_data['context'][:200]}...")
        else:
            print("QA 데이터 추출 실패 - 응답 형식 확인 필요")
            print(f"응답 내용: {response}")
    else:
        print("❌ API 호출 실패 - 위의 에러 메시지를 확인하세요")
else:
    print("⚠️ ETRI API 키가 설정되지 않았습니다!")
    print()
    print("=== API 키 발급 방법 ===")
    print("1. https://epretx.etri.re.kr 접속")
    print("2. 회원가입 후 로그인")
    print("3. API 메뉴에서 '위키백과 QA API' 선택")
    print("4. API 키 발급 신청")
    print("5. 발급받은 키를 위 셀의 ETRI_ACCESS_KEY 변수에 입력")
    print()
    print("⚠️ IP 제한: API 설정에서 현재 IP가 허용되어 있는지 확인하세요.")


✓ API 키 설정됨: d4a79a0b-6...
✓ API URL: http://epretx.etri.re.kr:8000/api/WikiQA/
--------------------------------------------------
테스트 질문: 대한민국의 수도는 어디인가요?
--------------------------------------------------
✅ API 호출 성공!
정답: 서울특별시
신뢰도: 2.0427
위키 제목: 수도
문맥: - ↑ 다만 서울특별시는 서울특별시행정특례에관한법률(일부개정 1995.12.6 법률 5000호)에 따라 대한민국의 수도로서 지위를 가지고 있었다. ...


## 2.4. 외부 데이터셋 생성 (ETRI API 활용)


In [49]:
def collect_etri_qa_data(
    questions: List[str],
    etri_client: ETRIWikiQA,
    engine_type: str = "hybridqa",
    delay: float = 0.2,
    min_confidence: float = 0.5
) -> List[Dict]:
    """
    ETRI API를 사용하여 QA 데이터 수집
    
    Args:
        questions: 질문 리스트
        etri_client: ETRI API 클라이언트
        engine_type: 엔진 타입
        delay: API 호출 간 딜레이 (초)
        min_confidence: 최소 신뢰도 임계값
    
    Returns:
        수집된 QA 데이터 리스트
    """
    collected_data = []
    
    for idx, question in enumerate(tqdm(questions, desc="데이터 수집")):
        try:
            response = etri_client.query(question, engine_type)
            
            if response:
                qa_data = etri_client.extract_qa_data(response)
                
                if qa_data and qa_data['confidence'] >= min_confidence:
                    # answer_start 계산
                    answer = qa_data['answer']
                    context = qa_data['context']
                    answer_start = context.find(answer)
                    
                    if answer_start != -1:  # context에서 answer를 찾은 경우만
                        collected_data.append({
                            'id': f"etri-{idx:05d}",
                            'question': question,
                            'context': context,
                            'answers': {
                                'text': [answer],
                                'answer_start': [answer_start]
                            },
                            'title': qa_data['wiki_title'],
                            'confidence': qa_data['confidence']
                        })
            
            # API 호출 제한을 위한 딜레이
            time.sleep(delay)
            
        except Exception as e:
            print(f"질문 처리 실패 [{idx}]: {e}")
            continue
    
    return collected_data


# 기존 데이터셋에서 질문 로드 (ETRI API 호출용)
original_data_path = project_root / "data" / "train_dataset"
if original_data_path.exists():
    original_datasets = load_from_disk(str(original_data_path))
    original_questions = original_datasets['train']['question']
    print(f"원본 데이터셋에서 {len(original_questions)}개의 질문 로드")
else:
    print("원본 데이터셋을 찾을 수 없습니다. 샘플 질문을 사용합니다.")
    original_questions = [
        "대한민국의 수도는?",
        "한글을 창제한 왕은?",
        "지구에서 가장 높은 산은?",
    ]


원본 데이터셋에서 3952개의 질문 로드


In [ ]:
# ETRI 데이터 로드 (00_etri_data_collection.ipynb에서 수집한 데이터)
etri_data_path = data_dir / "etri_qa_dataset.json"

if etri_data_path.exists():
    with open(etri_data_path, 'r', encoding='utf-8') as f:
        etri_qa_data = json.load(f)
    print(f"✅ ETRI 데이터 로드 완료: {len(etri_qa_data)}개")
else:
    print("❌ ETRI 데이터 파일이 없습니다!")
    print(f"   경로: {etri_data_path}")
    print("\n📝 먼저 00_etri_data_collection.ipynb를 실행하여 데이터를 수집하세요.")
    etri_qa_data = []


⚠️ API 키가 설정되지 않아 데이터 수집을 건너뜁니다.
기존에 수집된 데이터 파일이 있는지 확인합니다...
기존 데이터 파일이 없습니다. API 키를 설정하고 데이터를 수집하세요.


## 2.5. ETRI 데이터셋 분석


In [51]:
# ETRI 데이터셋 분석
if etri_qa_data:
    print("=== ETRI 데이터셋 분석 ===\n")
    print(f"총 샘플 수: {len(etri_qa_data)}")
    
    # 샘플 확인
    print("\n--- 샘플 데이터 ---")
    sample = etri_qa_data[0]
    print(f"ID: {sample['id']}")
    print(f"Question: {sample['question']}")
    print(f"Answer: {sample['answers']['text'][0]}")
    print(f"Context (처음 200자): {sample['context'][:200]}...")
    print(f"Title: {sample['title']}")
    print(f"Confidence: {sample['confidence']:.4f}")
    
    # 통계
    answer_lengths = [len(d['answers']['text'][0]) for d in etri_qa_data]
    context_lengths = [len(d['context']) for d in etri_qa_data]
    confidences = [d['confidence'] for d in etri_qa_data]
    
    print("\n--- 통계 ---")
    print(f"평균 정답 길이: {np.mean(answer_lengths):.1f}자")
    print(f"평균 문맥 길이: {np.mean(context_lengths):.1f}자")
    print(f"평균 신뢰도: {np.mean(confidences):.4f}")
else:
    print("ETRI 데이터가 없습니다. 데이터를 먼저 수집하세요.")


ETRI 데이터가 없습니다. 데이터를 먼저 수집하세요.


## 2.6. 데이터셋 준비 (Train/Validation 분할)


In [52]:
def prepare_etri_dataset(etri_data: List[Dict], val_ratio: float = 0.1) -> DatasetDict:
    """
    ETRI 데이터를 HuggingFace Dataset 형식으로 변환
    
    Args:
        etri_data: ETRI QA 데이터 리스트
        val_ratio: 검증 데이터 비율
    
    Returns:
        DatasetDict (train, validation)
    """
    # 데이터 셔플
    random.shuffle(etri_data)
    
    # Train/Validation 분할
    val_size = int(len(etri_data) * val_ratio)
    train_data = etri_data[val_size:]
    val_data = etri_data[:val_size]
    
    # Dataset 형식으로 변환 (confidence 필드 제거)
    def clean_data(data_list):
        return [{k: v for k, v in d.items() if k != 'confidence'} for d in data_list]
    
    train_dataset = Dataset.from_list(clean_data(train_data))
    val_dataset = Dataset.from_list(clean_data(val_data))
    
    return DatasetDict({
        'train': train_dataset,
        'validation': val_dataset
    })


# ETRI 데이터셋 준비
if etri_qa_data:
    etri_datasets = prepare_etri_dataset(etri_qa_data, val_ratio=0.1)
    print(f"ETRI Train 데이터: {len(etri_datasets['train'])} samples")
    print(f"ETRI Validation 데이터: {len(etri_datasets['validation'])} samples")
    print(f"\n데이터셋 컬럼: {etri_datasets['train'].column_names}")
else:
    print("ETRI 데이터가 없어 데이터셋을 생성할 수 없습니다.")


ETRI 데이터가 없어 데이터셋을 생성할 수 없습니다.


## 2.7. 모델 및 토크나이저 설정


In [53]:
# 모델 설정
MODEL_NAME = "klue/bert-base"
MAX_SEQ_LENGTH = 384
DOC_STRIDE = 128

# 토크나이저 및 모델 로드
print(f"모델 로드 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model_config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME, config=model_config)

print(f"토크나이저 vocab size: {tokenizer.vocab_size}")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")


모델 로드 중: klue/bert-base


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


토크나이저 vocab size: 32000
모델 파라미터 수: 110,028,290


## 2.8. 데이터 전처리


In [54]:
# 데이터 전처리 함수 (Baseline과 동일)
def prepare_train_features(examples):
    """학습 데이터 전처리"""
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized["start_positions"].append(cls_index)
                tokenized["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized["end_positions"].append(token_end_index + 1)

    return tokenized


def prepare_validation_features(examples):
    """검증 데이터 전처리"""
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_index])
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized


In [55]:
# 데이터 전처리 수행
if etri_qa_data:
    print("ETRI 학습 데이터 전처리 중...")
    etri_train_dataset = etri_datasets['train'].map(
        prepare_train_features,
        batched=True,
        remove_columns=etri_datasets['train'].column_names
    )

    print("ETRI 검증 데이터 전처리 중...")
    etri_validation_dataset = etri_datasets['validation'].map(
        prepare_validation_features,
        batched=True,
        remove_columns=etri_datasets['validation'].column_names
    )

    print(f"\n전처리된 ETRI 학습 데이터 샘플 수: {len(etri_train_dataset)}")
    print(f"전처리된 ETRI 검증 데이터 샘플 수: {len(etri_validation_dataset)}")
else:
    print("ETRI 데이터가 없어 전처리를 건너뜁니다.")


ETRI 데이터가 없어 전처리를 건너뜁니다.


## 2.9. 학습 설정 및 실행


In [56]:
# 학습 인자 설정
training_args = TrainingArguments(
    output_dir=str(experiment_dir),
    do_train=True,
    do_eval=True,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

# Data Collator
data_collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if training_args.fp16 else None
)

print("학습 설정 완료")
print(f"  - Learning Rate: {training_args.learning_rate}")
print(f"  - Batch Size: {training_args.per_device_train_batch_size}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - FP16: {training_args.fp16}")


학습 설정 완료
  - Learning Rate: 3e-05
  - Batch Size: 16
  - Epochs: 3
  - FP16: True


In [57]:
# 후처리 및 메트릭 함수
def post_processing_function(examples, features, predictions, stage="eval"):
    """예측 결과 후처리"""
    predictions = postprocess_qa_predictions(
        examples=examples,
        features=features,
        predictions=predictions,
        max_answer_length=30,
        output_dir=str(experiment_dir),
    )
    
    formatted_predictions = [
        {"id": k, "prediction_text": v} for k, v in predictions.items()
    ]
    
    if stage == "predict":
        return formatted_predictions
    
    references = [
        {"id": ex["id"], "answers": ex["answers"]}
        for ex in etri_datasets['validation']
    ]
    
    return EvalPrediction(
        predictions=formatted_predictions,
        label_ids=references
    )

metric = evaluate.load("squad")

def compute_metrics(p: EvalPrediction):
    result = metric.compute(predictions=p.predictions, references=p.label_ids)
    # Trainer가 eval_ 접두사가 붙은 메트릭을 기대하므로 접두사 추가
    return {f"eval_{k}": v for k, v in result.items()}


In [58]:
# Trainer 초기화 및 학습 (ETRI 데이터가 있는 경우)
if etri_qa_data:
    trainer = QuestionAnsweringTrainer(
        model=model,
        args=training_args,
        train_dataset=etri_train_dataset,
        eval_dataset=etri_validation_dataset,
        eval_examples=etri_datasets['validation'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        post_process_function=post_processing_function,
        compute_metrics=compute_metrics,
    )
    
    print("Trainer 초기화 완료")
    
    # 학습 실행
    print("=" * 50)
    print("ETRI 데이터셋 학습 시작")
    print("=" * 50)
    
    train_result = trainer.train()
    
    # 모델 저장
    trainer.save_model()
    
    # 학습 결과 저장
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    
    print("\n학습 완료!")
    print(f"Train Loss: {metrics.get('train_loss', 'N/A'):.4f}")
else:
    print("ETRI 데이터가 없어 학습을 건너뜁니다.")


ETRI 데이터가 없어 학습을 건너뜁니다.


## 2.10. 평가 및 결과 분석


In [59]:
# 평가 실행
if etri_qa_data:
    print("=" * 50)
    print("ETRI 데이터셋 평가 시작")
    print("=" * 50)

    eval_metrics = trainer.evaluate()

    # 평가 결과 저장
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)

    print("\n=== ETRI 데이터셋 평가 결과 ===")
    print(f"Exact Match (EM): {eval_metrics.get('eval_exact_match', 'N/A'):.2f}")
    print(f"F1 Score: {eval_metrics.get('eval_f1', 'N/A'):.2f}")
    
    # 결과 요약 저장
    results_summary = {
        "experiment": "etri_only",
        "model": MODEL_NAME,
        "train_samples": len(etri_datasets['train']),
        "eval_samples": len(etri_datasets['validation']),
        "epochs": training_args.num_train_epochs,
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "eval_exact_match": eval_metrics.get('eval_exact_match'),
        "eval_f1": eval_metrics.get('eval_f1'),
    }

    results_path = experiment_dir / "results_summary.json"
    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(results_summary, f, ensure_ascii=False, indent=2)
    print(f"\n결과 요약이 저장되었습니다: {results_path}")
else:
    print("ETRI 데이터가 없어 평가를 건너뜁니다.")


ETRI 데이터가 없어 평가를 건너뜁니다.


## 2.11. 결과 시각화


In [60]:
# 결과 시각화
if etri_qa_data:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # 메트릭 바 차트
    metrics_names = ['Exact Match', 'F1 Score']
    metrics_values = [
        eval_metrics.get('eval_exact_match', 0),
        eval_metrics.get('eval_f1', 0)
    ]

    colors = ['#e67e22', '#16a085']
    bars = axes[0].bar(metrics_names, metrics_values, color=colors)
    axes[0].set_ylabel('Score')
    axes[0].set_title('ETRI 데이터셋 모델 성능')
    axes[0].set_ylim(0, 100)

    for bar, val in zip(bars, metrics_values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=12)

    # 데이터셋 정보
    dataset_info = ['Train', 'Validation']
    dataset_sizes = [len(etri_datasets['train']), len(etri_datasets['validation'])]

    axes[1].pie(dataset_sizes, labels=dataset_info, autopct='%1.1f%%',
                colors=['#3498db', '#9b59b6'], startangle=90)
    axes[1].set_title('ETRI 데이터셋 구성')

    plt.tight_layout()
    plt.savefig(experiment_dir / "etri_results.png", dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n그래프 저장 완료: {experiment_dir / 'etri_results.png'}")
else:
    print("ETRI 데이터가 없어 시각화를 건너뜁니다.")


ETRI 데이터가 없어 시각화를 건너뜁니다.
